In [18]:
import sys
from pathlib import Path

In [19]:
project_root = Path("..").resolve()
sys.path.append(str(project_root))

In [20]:
from sqlalchemy.orm import Session
from sqlalchemy import select
from api.events.models import Event
from api.models.database import engine, get_db
from pprint import pprint

In [21]:
query_stmt = select(Event).where(Event.id == 1)

print(
    query_stmt.compile(
        engine,
        compile_kwargs={"literal_binds": True}
    )
)
with Session(engine) as sess:
    event = sess.execute(query_stmt).scalar_one_or_none()
    # compiled_query = sess.execute(query_stmt).scalar_one_or_none()
    print(event)

    print("")
    print(str(query_stmt))

SELECT [Events].id, [Events].page, [Events].description, [Events].[userId], [Events].[sessionId], [Events].[ipAddress], [Events].[userAgent], [Events].referrer, [Events].[createdDate] 
FROM [Events] 
WHERE [Events].id = 1

SELECT "Events".id, "Events".page, "Events".description, "Events"."userId", "Events"."sessionId", "Events"."ipAddress", "Events"."userAgent", "Events".referrer, "Events"."createdDate" 
FROM "Events" 
WHERE "Events".id = :id_1


In [ ]:
from timescaledb .hyperfunctions import time_bucket
from sqlalchemy import func, select, cast, Date

# bucket = time_bucket('1 day', Event.createdDate).label('bucket')
# query_stmt = (select
#                     (cast(Event.createdDate,Date).label('bucket'),
#                     Event.page,
#                     func.count().label('count')
#                     ).group_by
#                     (cast(Event.createdDate, Date), 
#                                Event.page)
#                                .order_by
#                                (cast(Event.createdDate. Date).desc())
#             )

query_stmt = (
    select(
        cast(Event.createdDate, Date).label("bucket"),
        Event.page,
        func.count().label("count"),
    )
    .group_by(
        cast(Event.createdDate, Date),
        Event.page,
    )
    .order_by(
        cast(Event.createdDate, Date).desc()
    )
)

print(
    query_stmt.compile(
        engine,
        compile_kwargs={"literal_binds": True}
    )
)
with Session(engine) as sess:
    event = sess.execute(query_stmt).all()
    # compiled_query = sess.execute(query_stmt).scalar_one_or_none()

    pprint(event)

    print("")
    # print(str(query_stmt))


SELECT CAST([Events].[createdDate] AS DATE) AS bucket, [Events].page, count(*) AS count 
FROM [Events] GROUP BY CAST([Events].[createdDate] AS DATE), [Events].page ORDER BY CAST([Events].[createdDate] AS DATE) DESC
[(datetime.date(2026, 7, 29), 'Checkout', 1),
 (datetime.date(2026, 7, 29), 'Contact', 1),
 (datetime.date(2026, 7, 29), 'Dashboard', 1),
 (datetime.date(2026, 7, 29), 'Home', 1),
 (datetime.date(2026, 7, 29), 'ksdlsklds', 1),
 (datetime.date(2026, 7, 29), 'Login', 1),
 (datetime.date(2026, 7, 29), 'Logout', 1),
 (datetime.date(2026, 7, 29), 'Notifications', 1),
 (datetime.date(2026, 7, 29), 'Orders', 1),
 (datetime.date(2026, 7, 29), 'Products', 1),
 (datetime.date(2026, 7, 29), 'Profile', 1),
 (datetime.date(2026, 7, 29), 'Reports', 1),
 (datetime.date(2026, 7, 29), 'Settings', 1),
 (datetime.date(2026, 7, 29), 'Support', 1),
 (datetime.date(2026, 7, 29), 'what page', 1)]

